In [2]:
import torch
import testdata
from _em import _EM_step_full_stable, fit_EM_iter

In [3]:
%load_ext autoreload

In [4]:
%autoreload 2

Generate fake data

In [5]:
params = {
    'd': 10, 
    'k': [5, 3, 4], 
    'p': [15, 13, 14], 
    'n': 5000,
    'sigsq': [0.3, 0.7, 0.5]
}

Y, W, L, Phi = testdata.simulate_data(params, private_var=True, verbose=True)
W_init, L_init, Phi_init = testdata.initialize_params(W, L, Phi, private_var=True)

Y = Y.T

Y = WZ + LX + E


Complete data case

In [8]:
metrics = {
    'WWt_corr': [],
    'LLt_corr': [],
    'Phi_corr': [],
}
problematic_phi = 0

for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=True)
    Y = Y.T

    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=True)
    Sigma_hat = Y.T @ Y / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    LL = (torch.block_diag(*L_true)) @ (torch.block_diag(*L_true).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, L_new, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute=False
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    LL_test = (torch.block_diag(*L_new)) @ (torch.block_diag(*L_new).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    metrics['LLt_corr'].append(torch.corrcoef(
        torch.stack([LL.flatten(), LL_test.flatten()], dim=0)
    )[0,1].item())
    phi_corr = torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item()
    if phi_corr < 0.9: problematic_phi += 1
    metrics['Phi_corr'].append(phi_corr)

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


1000 simulations done in 2m 7.2s

In [9]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9974
	Min: 0.9939
	Max: 0.9987
LLt_corr
	Mean: 0.9994
	Min: 0.9983
	Max: 0.9998
Phi_corr
	Mean: 0.9876
	Min: 0.9438
	Max: 0.9962


In [10]:
print(f"Percent of simulation runs with corr(Phi, Phi_init) < 0.9: {problematic_phi/1000 * 100}%")

Percent of simulation runs with corr(Phi, Phi_init) < 0.9: 0.0%


Testing for missing data case (each sample missing max one mode)

In [12]:
metrics = {
    'WWt_corr': [],
    'LLt_corr': [],
    'Phi_corr': [],
}
problematic_phi = 0

for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=True)
    Y = Y.T

    # insert missing data; missing modes do not overlap
    Y[1000:1025, :params['p'][0]] = float('nan')
    Y[1025:1075, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
    Y[1075:1100, params['p'][0]+params['p'][1]:] = float('nan')

    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=True)
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    LL = (torch.block_diag(*L_true)) @ (torch.block_diag(*L_true).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, L_new, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    LL_test = (torch.block_diag(*L_new)) @ (torch.block_diag(*L_new).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    metrics['LLt_corr'].append(torch.corrcoef(
        torch.stack([LL.flatten(), LL_test.flatten()], dim=0)
    )[0,1].item())
    phi_corr = torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item()
    if phi_corr < 0.9: problematic_phi += 1
    metrics['Phi_corr'].append(phi_corr)

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


1000 simulations took ~ 2m 34.9s to run

In [13]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9974
	Min: 0.9944
	Max: 0.9987
LLt_corr
	Mean: 0.9993
	Min: 0.9979
	Max: 0.9997
Phi_corr
	Mean: 0.9515
	Min: 0.6754
	Max: 0.985


In [16]:
print(f"Percent of simulation runs with corr(Phi, Phi_init) < 0.9: {round(problematic_phi/1000 * 100, 2)}%")

Percent of simulation runs with corr(Phi, Phi_init) < 0.9: 2.9%


Testing for missing data (samples may be missing up to 2/3 modes)

In [17]:
metrics = {
    'WWt_corr': [],
    'LLt_corr': [],
    'Phi_corr': [],
}

for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=True)
    Y = Y.T

    # insert missing data; missing modes overlap
    Y[990:1025, :params['p'][0]] = float('nan')
    Y[1020:1070, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
    Y[1065:1100, params['p'][0]+params['p'][1]:] = float('nan')
    Y[980:1000, params['p'][0]+params['p'][1]:] = float('nan')

    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=True)
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    LL = (torch.block_diag(*L_true)) @ (torch.block_diag(*L_true).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, L_new, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    LL_test = (torch.block_diag(*L_new)) @ (torch.block_diag(*L_new).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    metrics['LLt_corr'].append(torch.corrcoef(
        torch.stack([LL.flatten(), LL_test.flatten()], dim=0)
    )[0,1].item())
    phi_corr = torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item()
    if phi_corr < 0.9: problematic_phi += 1
    metrics['Phi_corr'].append(phi_corr)

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


1000 simulations ran in ~ 2m 29.6s

In [18]:
for k, v in metrics.items():
    v = torch.tensor(v)
    print(k)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9974
	Min: 0.9944
	Max: 0.9986
LLt_corr
	Mean: 0.9993
	Min: 0.9975
	Max: 0.9997
Phi_corr
	Mean: 0.9394
	Min: 0.5871
	Max: 0.9839


In [20]:
print(f"Percent of simulation runs with corr(Phi, Phi_init) < 0.9: {round(problematic_phi/1000 * 100, 2)}%")

Percent of simulation runs with corr(Phi, Phi_init) < 0.9: 8.8%
